
### Folder 07 — File 04: Gold Payment Metrics:


This notebook builds **payment analytics** using **Silver payments** and **orders**.

It calculates:

- Total payment transactions.
- Successful, failed, pending and refunded payment counts.
- Payment success rate.
- Payment failure rate.
- Total successful payment amount.
- Total refunded amount.
- Net payment amount.
- Payment-method performance.
- Daily payment trends.
- Order-level payment coverage.

**Important:**

- A payment success rate is calculated using payment attempts, not orders. 
- An order can have multiple payment attempts, so these are different business metrics.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

PAYMENTS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_payments"
)

ORDERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
)

GOLD_PAYMENT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_payment_metrics"
)

GOLD_METHOD_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_payment_method_performance"
)

GOLD_ORDER_PAYMENT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_order_payment_summary"
)

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

RUN_TIMESTAMP = datetime.now()

print("Run ID:", RUN_ID)


## 3. Validate source tables

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)

SOURCE_TABLES = [
    PAYMENTS_TABLE,
    ORDERS_TABLE
]

for table_name in SOURCE_TABLES:

    if not spark.catalog.tableExists(table_name):
        raise Exception(
            f"Missing source table: {table_name}"
        )

print("All source tables are available.")


## 4. Load Silver datasets

payments_df = spark.table(PAYMENTS_TABLE)

orders_df = spark.table(ORDERS_TABLE)

print("Payments:", payments_df.count())
print("Orders:", orders_df.count())


## 5. Validate required columns

required_payment_columns = [
    "payment_id",
    "order_id",
    "payment_method",
    "payment_status",
    "amount",
    "payment_timestamp"
]

required_order_columns = [
    "order_id",
    "customer_id",
    "order_timestamp",
    "order_status",
    "order_total",
    "currency"
]

for column_name in required_payment_columns:

    if column_name not in payments_df.columns:
        raise Exception(
            f"Missing payment column: {column_name}"
        )

for column_name in required_order_columns:

    if column_name not in orders_df.columns:
        raise Exception(
            f"Missing order column: {column_name}"
        )


## 6. Clean payment data

payments_clean_df = (
    payments_df
    .withColumn(
        "payment_id",
        F.trim(F.col("payment_id"))
    )
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "payment_method",
        F.upper(F.trim(F.col("payment_method")))
    )
    .withColumn(
        "payment_status",
        F.upper(F.trim(F.col("payment_status")))
    )
    .withColumn(
        "amount",
        F.col("amount").cast("decimal(18,2)")
    )
    .withColumn(
        "payment_timestamp",
        F.to_timestamp("payment_timestamp")
    )
    .withColumn(
        "payment_date",
        F.to_date("payment_timestamp")
    )
    .filter(
        F.col("payment_id").isNotNull()
        & (F.col("payment_id") != "")
        & F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
        & F.col("payment_method").isNotNull()
        & F.col("payment_status").isin(
            "SUCCESS",
            "PENDING",
            "FAILED",
            "REFUNDED"
        )
        & F.col("amount").isNotNull()
        & (F.col("amount") >= 0)
        & F.col("payment_timestamp").isNotNull()
    )
)


## 7. Deduplicate payment records

if "_ingestion_timestamp" in payments_clean_df.columns:

    payment_window = (
        Window
        .partitionBy("payment_id")
        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )
    )

else:

    payment_window = (
        Window
        .partitionBy("payment_id")
        .orderBy(
            F.col("payment_timestamp").desc()
        )
    )

payments_deduplicated_df = (
    payments_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(payment_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


## 8. Clean order data

orders_clean_df = (
    orders_df
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )
    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )
    .withColumn(
        "order_timestamp",
        F.to_timestamp("order_timestamp")
    )
    .withColumn(
        "order_date",
        F.to_date("order_timestamp")
    )
    .withColumn(
        "order_total",
        F.col("order_total").cast("decimal(18,2)")
    )
    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency")))
    )
    .filter(
        F.col("order_id").isNotNull()
        & F.col("order_timestamp").isNotNull()
        & F.col("order_total").isNotNull()
        & (F.col("order_total") >= 0)
        & F.col("currency").isNotNull()
    )
)


## 9. Deduplicate orders

if "_ingestion_timestamp" in orders_clean_df.columns:

    order_window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )
    )

else:

    order_window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("order_timestamp").desc()
        )
    )

orders_deduplicated_df = (
    orders_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(order_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


## 10. Create payment transaction metrics

# Each payment record represents one payment transaction or attempt.

# Status metrics are calculated at the payment transaction level.

payment_metrics_df = (
    payments_deduplicated_df
    .groupBy(
        "payment_date"
    )
    .agg(

        F.countDistinct("payment_id").alias(
            "total_payment_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                1
            ).otherwise(0)
        ).alias(
            "successful_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "PENDING",
                1
            ).otherwise(0)
        ).alias(
            "pending_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                1
            ).otherwise(0)
        ).alias(
            "refunded_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "successful_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "refunded_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "FAILED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "failed_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "PENDING",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "pending_payment_amount"
        )
    )
)

## 11. Calculate payment success and failure rates

# Rates use all recorded payment attempts as the denominator.
# Refunded transactions are not counted as successful in this calculation.

payment_metrics_df = (
    payment_metrics_df
    .withColumn(
        "payment_success_rate",
        F.when(
            F.col("total_payment_transactions") > 0,
            F.round(
                F.col("successful_transactions")
                / F.col("total_payment_transactions") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "payment_failure_rate",
        F.when(
            F.col("total_payment_transactions") > 0,
            F.round(
                F.col("failed_transactions")
                / F.col("total_payment_transactions") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "refund_transaction_rate",
        F.when(
            F.col("total_payment_transactions") > 0,
            F.round(
                F.col("refunded_transactions")
                / F.col("total_payment_transactions") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "net_payment_amount",
        (
            F.col("successful_payment_amount")
            - F.col("refunded_payment_amount")
        ).cast("decimal(18,2)")
    )
)


## 12. Create payment-method performance

# Compare transaction success and failure across payment methods.

payment_method_df = (
    payments_deduplicated_df
    .groupBy(
        "payment_method"
    )
    .agg(

        F.countDistinct("payment_id").alias(
            "total_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                1
            ).otherwise(0)
        ).alias(
            "successful_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "PENDING",
                1
            ).otherwise(0)
        ).alias(
            "pending_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                1
            ).otherwise(0)
        ).alias(
            "refunded_transactions"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "successful_payment_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "refunded_payment_amount"
        )
    )
    .withColumn(
        "success_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.round(
                F.col("successful_transactions")
                / F.col("total_transactions") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "failure_rate",
        F.when(
            F.col("total_transactions") > 0,
            F.round(
                F.col("failed_transactions")
                / F.col("total_transactions") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "net_payment_amount",
        (
            F.col("successful_payment_amount")
            - F.col("refunded_payment_amount")
        ).cast("decimal(18,2)")
    )
)


## 13. Build order-level payment summary

# Aggregate payment attempts by order before joining to orders.
# This prevents one-to-many payment joins from duplicating order values.

order_payment_aggregation_df = (
    payments_deduplicated_df
    .groupBy("order_id")
    .agg(

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "successful_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "REFUNDED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "refunded_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "PENDING",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "pending_amount"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "FAILED",
                F.col("amount")
            ).otherwise(F.lit(0))
        ).cast("decimal(18,2)").alias(
            "failed_amount"
        ),

        F.countDistinct("payment_id").alias(
            "payment_attempts"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                1
            ).otherwise(0)
        ).alias(
            "successful_attempts"
        )
    )
)


## 14. Join payment aggregates with orders

# Keep order value separate from payment amounts.
# Cancelled orders remain visible for reconciliation.

order_payment_summary_df = (
    orders_deduplicated_df.alias("orders")
    .join(
        order_payment_aggregation_df.alias("payments"),
        F.col("orders.order_id")
        == F.col("payments.order_id"),
        "left"
    )
    .select(
        F.col("orders.order_id"),
        F.col("orders.customer_id"),
        F.col("orders.order_date"),
        F.col("orders.order_status"),
        F.col("orders.currency"),
        F.col("orders.order_total"),

        F.coalesce(
            F.col("payments.successful_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("successful_amount"),

        F.coalesce(
            F.col("payments.refunded_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("refunded_amount"),

        F.coalesce(
            F.col("payments.pending_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("pending_amount"),

        F.coalesce(
            F.col("payments.failed_amount"),
            F.lit(0).cast("decimal(18,2)")
        ).alias("failed_amount"),

        F.coalesce(
            F.col("payments.payment_attempts"),
            F.lit(0)
        ).alias("payment_attempts"),

        F.coalesce(
            F.col("payments.successful_attempts"),
            F.lit(0)
        ).alias("successful_attempts")
    )
    .withColumn(
        "net_payment_amount",
        (
            F.col("successful_amount")
            - F.col("refunded_amount")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "payment_coverage_status",
        F.when(
            F.col("payment_attempts") == 0,
            "NO_PAYMENT_RECORD"
        )
        .when(
            F.col("successful_amount") == 0,
            "NO_SUCCESSFUL_PAYMENT"
        )
        .when(
            F.col("net_payment_amount") < F.col("order_total"),
            "UNDERPAID_OR_PARTIALLY_REFUNDED"
        )
        .when(
            F.col("net_payment_amount") > F.col("order_total"),
            "OVERPAID"
        )
        .otherwise("MATCHED")
    )
)

## 15. Calculate order payment coverage metrics

order_payment_coverage_df = (
    order_payment_summary_df
    .groupBy(
        "order_date",
        "currency"
    )
    .agg(

        F.countDistinct("order_id").alias(
            "total_orders"
        ),

        F.sum(
            F.when(
                F.col("payment_coverage_status") == "MATCHED",
                1
            ).otherwise(0)
        ).alias(
            "matched_orders"
        ),

        F.sum(
            F.when(
                F.col("payment_coverage_status") == "NO_PAYMENT_RECORD",
                1
            ).otherwise(0)
        ).alias(
            "orders_without_payment_records"
        ),

        F.sum(
            F.when(
                F.col("payment_coverage_status") == "NO_SUCCESSFUL_PAYMENT",
                1
            ).otherwise(0)
        ).alias(
            "orders_without_successful_payment"
        ),

        F.sum(
            F.when(
                F.col("payment_coverage_status") == "UNDERPAID_OR_PARTIALLY_REFUNDED",
                1
            ).otherwise(0)
        ).alias(
            "underpaid_or_partially_refunded_orders"
        ),

        F.sum(
            F.when(
                F.col("payment_coverage_status") == "OVERPAID",
                1
            ).otherwise(0)
        ).alias(
            "overpaid_orders"
        )
    )
    .withColumn(
        "matched_order_rate",
        F.when(
            F.col("total_orders") > 0,
            F.round(
                F.col("matched_orders")
                / F.col("total_orders") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)


## 16. Add audit columns

payment_metrics_df = (
    payment_metrics_df
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

payment_method_df = (
    payment_method_df
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

order_payment_summary_df = (
    order_payment_summary_df
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

order_payment_coverage_df = (
    order_payment_coverage_df
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)


## 17. Display daily payment metrics

display(
    payment_metrics_df.orderBy(
        F.col("payment_date").desc()
    )
)


## 18. Display payment method performance

display(
    payment_method_df.orderBy(
        F.col("success_rate").desc()
    )
)


## 19. Display order payment coverage

display(
    order_payment_coverage_df.orderBy(
        F.col("order_date").desc()
    )
)


## 20. Save Gold daily payment metrics

(
    payment_metrics_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_PAYMENT_TABLE)
)

print("Daily payment table created:", GOLD_PAYMENT_TABLE)


## 21. Save Gold payment method performance

(
    payment_method_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_METHOD_TABLE)
)

print("Payment method table created:", GOLD_METHOD_TABLE)


## 22. Save Gold order payment summary

(
    order_payment_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ORDER_PAYMENT_TABLE)
)

print("Order payment table created:", GOLD_ORDER_PAYMENT_TABLE)


## 23. Final validation

gold_payment_df = spark.table(GOLD_PAYMENT_TABLE)

gold_method_df = spark.table(GOLD_METHOD_TABLE)

gold_order_payment_df = spark.table(GOLD_ORDER_PAYMENT_TABLE)

print("Daily payment records:", gold_payment_df.count())

print("Payment method records:", gold_method_df.count())

print("Order payment records:", gold_order_payment_df.count())

display(
    gold_payment_df
    .groupBy()
    .agg(
        F.sum("total_payment_transactions").alias(
            "total_transactions"
        ),
        F.sum("successful_transactions").alias(
            "successful_transactions"
        ),
        F.sum("failed_transactions").alias(
            "failed_transactions"
        ),
        F.sum("successful_payment_amount").alias(
            "successful_amount"
        ),
        F.sum("refunded_payment_amount").alias(
            "refunded_amount"
        ),
        F.sum("net_payment_amount").alias(
            "net_payment_amount"
        )
    )
)

print("=" * 65)
print("GOLD PAYMENT METRICS COMPLETED")
print("=" * 65)

print("Daily payment table:", GOLD_PAYMENT_TABLE)
print("Payment method table:", GOLD_METHOD_TABLE)
print("Order payment table:", GOLD_ORDER_PAYMENT_TABLE)



## Databricks SQL dashboard queries:


### Query 1 — Daily payment trends

In [0]:
%sql

SELECT
    payment_date,
    total_payment_transactions,
    successful_transactions,
    failed_transactions,
    pending_transactions,
    refunded_transactions,
    payment_success_rate,
    payment_failure_rate,
    net_payment_amount
FROM workspace.ecommerce_gold.gold_payment_metrics
ORDER BY payment_date;


### Payment method comparison

In [0]:
%sql

SELECT
    payment_method,
    total_transactions,
    successful_transactions,
    failed_transactions,
    success_rate,
    failure_rate,
    net_payment_amount
FROM workspace.ecommerce_gold.gold_payment_method_performance
ORDER BY success_rate DESC;


### Orders with payment issues

In [0]:
%sql

SELECT
    order_id,
    customer_id,
    order_total,
    successful_amount,
    refunded_amount,
    net_payment_amount,
    payment_coverage_status
FROM workspace.ecommerce_gold.gold_order_payment_summary
WHERE payment_coverage_status != 'MATCHED'
ORDER BY order_date DESC;


### Overall payment summary

In [0]:
%sql

SELECT
    SUM(total_payment_transactions) AS total_transactions,
    SUM(successful_transactions) AS successful_transactions,
    SUM(failed_transactions) AS failed_transactions,
    SUM(pending_transactions) AS pending_transactions,
    SUM(refunded_transactions) AS refunded_transactions,
    SUM(successful_payment_amount) AS successful_amount,
    SUM(refunded_payment_amount) AS refunded_amount,
    SUM(net_payment_amount) AS net_payment_amount
FROM workspace.ecommerce_gold.gold_payment_metrics;

In [0]:
%sql

DESCRIBE TABLE workspace.ecommerce_gold.gold_payment_method_performance;